In [2]:
from pathlib import Path
import json
from typing import Optional, Callable, Tuple
import random
from functools import partial
from copy import deepcopy

import cv2
import numpy as np

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.transforms import v2 as v2_transforms
from torchvision.transforms import functional as VF, InterpolationMode
from torchvision.io import read_image

from matplotlib import pyplot as plt
from tqdm import tqdm

from tire_vision.text.preprocessor.model import SidewallSegmentator
from tire_vision.text.preprocessor.unwrapper import SidewallUnwrapper
from tire_vision.config import SidewallSegmentatorConfig, SidewallUnwrapperConfig

In [3]:
cfg_segmentator = SidewallSegmentatorConfig()
cfg_unwrapper = SidewallUnwrapperConfig()

model = SidewallSegmentator(cfg_segmentator)
unwrapper = SidewallUnwrapper(cfg_unwrapper)

input_dir = Path("/Users/n-zagainov/kolobok/ml/data/annotations")
output_dir = Path("/Users/n-zagainov/kolobok/ml/data/annotations_unwrapped")

In [4]:
for img_path in input_dir.iterdir():
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask = model.detect(img)
    unwrapped = unwrapper.get_unwrapped_tire(img, mask)
    unwrapped = cv2.cvtColor(unwrapped, cv2.COLOR_RGB2BGR)
    output_dir.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_dir / img_path.name), unwrapped)